# Brasa U-Net: is it worth pursuing? (Colab GPU run)

Evaluates the **already-trained** U-Net (`model/unet.pt`, inside the bundle). No U-Net is retrained.

| Step | Question | Time (T4, rough) |
|---|---|---|
| 0 | Build the input tensors from `features.parquet` | ~5–10 min |
| 1 | Does a **simple per-pixel model with neighbourhood features** match the U-Net? If so, prefer it. | ~15 min |
| 2 | Which inputs does the U-Net use? (permutation importance) | ~10 min |
| 3 | Full comparison: 2019–2024 and **2025** (drift), per-year bias, bootstrap CIs, fire regimes, top countries. | ~5 min |

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*.
2. Put `brasa-colab.tar` (≈210 MB) in Google Drive at **`MyDrive/brasa/brasa-colab.tar`**. If it isn't there, the mount cell opens an upload dialog instead and saves a copy to Drive.
3. *Runtime → Run all*. Steps skip work whose output already exists, so after a disconnect just run all again.

Result tables and logs go to **`MyDrive/brasa/results/`**. The last cell downloads `brasa-results.zip`: send that back.

In [ ]:
# --- settings ---------------------------------------------------------------
BUNDLE = "/content/drive/MyDrive/brasa/brasa-colab.tar"
RESULTS = "/content/drive/MyDrive/brasa/results"     # persisted on Drive
WORK = "/content/work"                               # fast local disk (lost on disconnect, rebuilt automatically)
RUN_IMPORTANCE = True

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")
import os, shutil
os.makedirs(os.path.dirname(BUNDLE), exist_ok=True)
if not os.path.exists(BUNDLE):
    print("brasa-colab.tar not found on Drive: pick it in the upload dialog")
    up = files.upload()
    shutil.move(next(iter(up)), BUNDLE)      # keep a copy on Drive for next time
for d in (RESULTS, f"{RESULTS}/logs", f"{RESULTS}/tables", WORK, f"{WORK}/cells"):
    os.makedirs(d, exist_ok=True)

In [ ]:
# unpack code + data to local disk (≈30 s)
if not os.path.exists(f"{WORK}/brasa/ml/train.py"):
    !tar -xf "{BUNDLE}" -C "{WORK}"
!ls "{WORK}" "{WORK}/firms/derived/ml" "{WORK}/firms/derived/app"
REPO = f"{WORK}/brasa"
DER = f"{WORK}/firms/derived"
STACK = f"{DER}/ml/overlap"
%cd {REPO}

In [ ]:
# duckdb is the only dependency Colab lacks; torch / numpy / pandas / pyarrow are preinstalled
!pip -q install duckdb

## Step 0 – build the input tensors (~5–10 min, CPU)
Writes ≈3.7 GB of memmaps to local disk. Rebuilt automatically after a disconnect.

In [ ]:
if not os.path.exists(f"{STACK}/meta.json"):
    !python -I ml/build_tensor.py {DER} --years 2012-2025
else:
    print("tensors already built")

## Step 1 – trained U-Net vs. simple neighbourhood baselines
`ml/context_baseline.py` fits the same residual model as the U-Net (`log μ = log(4.1·M + 0.05) + g(x)`), but `g` only sees hand-built neighbourhood sums (1/3/9/25 cells, previous/current/next month): a linear version and a small MLP. If the MLP comes within ~0.01 of the U-Net at 5°, the network isn't buying much.

In [ ]:
MODEL = f"{REPO}/model/unet.pt"   # the trained model, shipped in the bundle

out = f"{WORK}/cells/unet_cells.parquet"
if not os.path.exists(out):
    !python -I ml/predict_cells.py {STACK} {MODEL} {out} --years 2019-2025
if not os.path.exists(f"{WORK}/cells/ctx_mlp_cells.parquet"):
    !python -I ml/context_baseline.py {STACK} {WORK}/cells 2>&1 | tee {RESULTS}/logs/context_baseline.log

In [ ]:
# quick verdict for step 1 (step 3 repeats it with everything saved)
!python -I ml/compare.py {DER} {WORK}/cells 2>&1 | tee {RESULTS}/logs/compare_step1.log

## Step 2 – permutation importance
How much the 5° error grows when each input group is scrambled. Large values = the network relies on it.

In [ ]:
if RUN_IMPORTANCE and not os.path.exists(f"{RESULTS}/tables/importance.json"):
    !python -I ml/evaluate.py {DER} {STACK} {MODEL} --importance --out {WORK}/eval 2>&1 | tee {RESULTS}/logs/evaluate.log
    for f in ("importance.json", "comparison.csv"):
        shutil.copy(f"{WORK}/eval/{f}", f"{RESULTS}/tables/{f}")
!cat {RESULTS}/tables/importance.json

## Step 3 – full comparison
Brasa M3, the U-Net, the two simple baselines and the NOAA-20 ceiling side by side.

In [ ]:
!python -I ml/compare.py {DER} {WORK}/cells 2>&1 | tee {RESULTS}/logs/compare.log
for f in ("scores.csv", "regimes.csv", "countries.csv", "bootstrap.json"):
    shutil.copy(f"{WORK}/cells/{f}", f"{RESULTS}/tables/{f}")

## How to read it

- **Simple baseline vs U-Net**: the `ctx_mlp - unet` line of the bootstrap. Interval entirely above 0 → the U-Net adds something beyond neighbourhood pooling, so keep it. Near 0 or below → prefer the simpler model.
- **vs Brasa**: `unet - Brasa M3` and `ctx_mlp - Brasa M3`: interval entirely below 0 = a real improvement.
- **Drift**: the *bias % by year* table, especially 2025.
- **Regimes**: the `MODIS saw nothing` row is the small-fire floor, the U-Net's known weak spot (−15% bias).
- **Importance**: which input groups the U-Net relies on. A large *neighbour months* or *MODIS climatology* value means context matters, not just this month's count.

In [ ]:
# bundle the small outputs to send back (no checkpoints or parquet)
!cd {RESULTS} && zip -qr /content/brasa-results.zip tables logs && ls -la /content/brasa-results.zip
from google.colab import files
files.download("/content/brasa-results.zip")